# HW1 Part IV Colab Runner

This notebook is the top-level Colab entrypoint for the Part IV parameterization sweep and sampling-step ablation. Core logic lives in Python scripts so the run can be resumed and debugged from saved logs, metrics, and checkpoints.

In [1]:
# Runtime Check
import os
import sys
import subprocess
import time
import shutil
from pathlib import Path

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("This Part IV runner expects a GPU runtime. Switch Colab to a GPU runtime first.")

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("GPU:", gpu_name)
print(f"GPU memory: {gpu_memory_gb:.1f} GB")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 39.5 GB


In [2]:
# Environment Setup
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/Eryc123Y/cmu-10799-diffusion.git"
REPO_DIR = Path("/content/cmu-10799-diffusion")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/cmu-10799-diffusion")
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "part4_runs" if USE_GOOGLE_DRIVE else Path("/content/part4_runs")

MAX_PARALLEL = 4
NUM_ITERATIONS = 100000
RUN_SMOKE_TEST = True
RUN_FULL_SWEEP = True
RUN_EVALUATION = True
RUN_ANALYSIS = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    cwd = Path(cwd) if cwd is not None else REPO_DIR
    print("\n$", " ".join(str(part) for part in cmd))
    return subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)

run([sys.executable, "-m", "pip", "install", "-q", "torch-fidelity", "datasets", "tqdm", "matplotlib", "pandas"], cwd=Path("/content"))
print("Output root:", OUTPUT_ROOT)

Mounted at /content/drive

$ /usr/bin/python3 -m pip install -q torch-fidelity datasets tqdm matplotlib pandas
Output root: /content/drive/MyDrive/cmu-10799-diffusion/part4_runs


In [ ]:
# Prepare Code
# Preferred: push the latest code to GitHub, then this cell clones/pulls it.
# Alternative: place HW1_CMU_10799_Spring_2026/hw1_code.zip in Drive and this cell will unzip it.
CODE_ZIP = DRIVE_PROJECT_DIR / "HW1_CMU_10799_Spring_2026" / "hw1_code.zip"

if REPO_DIR.exists() and (REPO_DIR / "train.py").exists():
    print("Using existing repo:", REPO_DIR)
elif CODE_ZIP.exists():
    print("Using code zip from Drive:", CODE_ZIP)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["unzip", "-q", "-o", str(CODE_ZIP), "-d", "/content"], cwd=Path("/content"))
    if not REPO_DIR.exists():
        candidates = [path for path in Path("/content").iterdir() if path.is_dir() and (path / "train.py").exists()]
        if not candidates:
            raise FileNotFoundError("Code zip did not produce a repo containing train.py")
        candidates[0].rename(REPO_DIR)
else:
    print("Cloning repo from GitHub:", REPO_URL)
    run(["git", "clone", REPO_URL, str(REPO_DIR)], cwd=Path("/content"))

required_paths = [
    "train.py",
    "sample.py",
    "src/methods/ddpm.py",
    "scripts/run_part4_param_sweep.py",
    "scripts/evaluate_part4_sweep.py",
    "scripts/analyze_part4_runs.py",
    "configs/part4/ddpm_epsilon_colab.yaml",
    "configs/part4/ddpm_x0_colab.yaml",
    "configs/part4/ddpm_v_colab.yaml",
    "configs/part4/ddpm_score_colab.yaml",
]
for relative_path in required_paths:
    path = REPO_DIR / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
print("Repo ready:", REPO_DIR)

In [ ]:
# Smoke Test: 4 parameterizations for 2k steps.
SMOKE_TIMESTAMP = time.strftime("smoke_%Y%m%d_%H%M%S")
SMOKE_SUMMARY = OUTPUT_ROOT / SMOKE_TIMESTAMP / "sweep_summary.json"

if RUN_SMOKE_TEST:
    run([
        sys.executable,
        "scripts/run_part4_param_sweep.py",
        "--output-root", OUTPUT_ROOT,
        "--timestamp", SMOKE_TIMESTAMP,
        "--max-parallel", "4",
        "--num-iterations", "2000",
        "--sample-every", "1000",
        "--save-every", "1000",
    ])
    print("Smoke summary:", SMOKE_SUMMARY)
else:
    print("Smoke test skipped.")

In [ ]:
# Full Parameterization Sweep
FULL_TIMESTAMP = time.strftime("full_%Y%m%d_%H%M%S")
FULL_RUN_ROOT = OUTPUT_ROOT / FULL_TIMESTAMP
FULL_SUMMARY = FULL_RUN_ROOT / "sweep_summary.json"

if RUN_FULL_SWEEP:
    run([
        sys.executable,
        "scripts/run_part4_param_sweep.py",
        "--output-root", OUTPUT_ROOT,
        "--timestamp", FULL_TIMESTAMP,
        "--max-parallel", str(MAX_PARALLEL),
        "--num-iterations", str(NUM_ITERATIONS),
        "--sample-every", "10000",
        "--save-every", "5000",
        "--resume-latest",
    ])
    print("Full sweep summary:", FULL_SUMMARY)
else:
    print("Full sweep skipped. Set FULL_SUMMARY manually before evaluation.")

In [ ]:
# Q6 KID Evaluation + Q7 Sampling Steps Ablation
if RUN_EVALUATION:
    if not FULL_SUMMARY.exists():
        raise FileNotFoundError(f"FULL_SUMMARY not found: {FULL_SUMMARY}")
    run([
        sys.executable,
        "scripts/evaluate_part4_sweep.py",
        "--sweep-summary", FULL_SUMMARY,
        "--num-samples", "1000",
        "--sample-batch-size", "32",
        "--real-batch-size", "64",
        "--real-num-workers", "0",
        "--kid-subset-size", "100",
        "--kid-subsets", "10",
        "--steps", "100", "300", "500", "700", "900", "1000",
    ])
else:
    print("Evaluation skipped.")

In [ ]:
# Analysis Figures
if RUN_ANALYSIS:
    run([
        sys.executable,
        "scripts/analyze_part4_runs.py",
        "--sweep-summary", FULL_SUMMARY,
        "--figures-dir", REPO_DIR / "HW1_CMU_10799_Spring_2026" / "figures",
    ])
else:
    print("Analysis skipped.")

In [ ]:
# Export Results
archive_base = OUTPUT_ROOT / f"part4_results_{FULL_TIMESTAMP}"
if FULL_RUN_ROOT.exists():
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=FULL_RUN_ROOT)
    print("Results zip:", archive_path)
else:
    print("Full run root not found:", FULL_RUN_ROOT)

print("\nReport values and figures to inspect:")
print("Q6 table:", FULL_RUN_ROOT / "evaluation" / "q6_parameterization" / "q6_kid_table.csv")
print("Q7 table:", FULL_RUN_ROOT / "evaluation" / "q7_sampling_steps" / "q7_kid_table.csv")
print("Figures:", REPO_DIR / "HW1_CMU_10799_Spring_2026" / "figures")
print("Summary:", FULL_SUMMARY)